# Maestra `compare()` quickstart

**Not executed in CI** (offline smoke-test coverage instead: `tests/test_public_api.py`, two sklearn dummies). Run this notebook interactively — in Colab or locally — with real internet/API access.

`compare()` is Maestra's arbiter as a generic DS tool: it honestly decides whether one sklearn-compatible pipeline measurably beats another, using the same paired, Nadeau-Bengio-corrected statistical test every internal Maestra gate uses (`docs/RESULTS.md`'s N1 hardening). No LLM call, and — deliberately — **no AutoGluon needed**: `compare()` only touches plain sklearn estimators.

Because this notebook never needs AutoGluon (a large, slow install pulling in PyTorch), we install Maestra with `--no-deps` below and add only the light dependencies `compare()` actually needs. Everything else in Maestra's `pyproject.toml` (AutoGluon, litellm, ...) stays uninstalled — this notebook's whole point is that `compare()` doesn't need them.

In [1]:
# Colab already ships pandas/numpy/scikit-learn. --no-deps skips maestra's own
# pyproject.toml dependencies (in particular autogluon.tabular[all], which pulls in PyTorch)
# since compare() never touches them.
!pip install --no-deps -q git+https://github.com/helenaschulz/maestra.git

zsh:1: /Users/helena.schulz.ext/code/maestra/.venv/bin/pip: bad interpreter: /Users/helena.schulz.ext/code/automl-agent/.venv/bin/python: no such file or directory


In [2]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from maestra import compare

# A small, built-in regression dataset (442 rows) -- no download, no API key.
data = load_diabetes(as_frame=True)
df = data.frame  # features + "target" column, already a single DataFrame
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [3]:
# Two arbitrary sklearn-compatible PIPELINES -- compare() only needs fit/predict(/predict_proba),
# which sklearn's own Pipeline already provides. Neither is AutoGluon; both are just sklearn.
baseline = LinearRegression()
candidate = Pipeline([
    ("scale", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("ridge", Ridge(alpha=1.0)),
])

result = compare(baseline, candidate, df, "target", cv=5, seeds=3)
print(result.summary())

**compare() verdict: no_improvement** (estimator.score, higher is better)
- a (baseline): 0.4812 · b: 0.4137
- mean delta (b − a, signed so +ve = b better): -0.0675
- minimum detectable effect: 0.1028
- 3 seed(s) × 5 fold(s), 15 paired observation(s)


`result.verdict` is one of `"improved"`, `"no_improvement"`, or `"underpowered"` — never a bare
number to eyeball. `result.mde` reports the minimum effect size this many seeds/folds could
even have detected, so an `"underpowered"`/`"no_improvement"` verdict is distinguishable from
"no effect" (see `docs/RESULTS.md`'s P0.2 MDE-Ausweis). `result.deltas` holds every pooled,
paired per-(seed, fold) delta if you want to inspect the spread yourself:

In [4]:
pd.Series(result.deltas, name="paired delta (candidate - baseline)").describe()

count    15.000000
mean     -0.067473
std       0.091384
min      -0.274120
25%      -0.114473
50%      -0.045564
75%      -0.005895
max       0.032254
Name: paired delta (candidate - baseline), dtype: float64